In [1]:
"""
exp02 Step 02: Daily Signal Check (PROTOCOL.md §11.1, stopping rule of the re-entry model)

Question: does any single daily feature move the share of positive 20-session forward returns AWAY FROM THE BASE RATE,
reliably, on training AND on the pooled validation quarters?

    - Reference = the base rate of each window (market drift alone cannot pass this test).
    - Two-sided: a bin reliably BELOW the base rate ("stay in cash") counts as much as one above it.
    - Bins: 5 quantile bins per feature, edges learned on training. Intervals: block bootstrap over calendar months.
    - A bin's pass counts only if its rows cover at least DAILY_SIGNAL_CHECK_MIN_MONTH_COUNT (6) calendar months in each
      window (new in exp02 1.0).
    - Windows: the 10-year training window of the 4th most recent fold, and the validation windows of the 4 most recent
      folds pooled together (about one year, all after that training window). The latest test window is NOT read.
      Those validation quarters are test windows of OLDER folds, so a design decision taken here counts as having seen them.

Verdict: STOP or pivot if no bin is a signal. One signal-check entry is written to the trial log.
Prerequisite: the raw data only. Outputs: store04_experiments/exp02_stop_reentry/step02_signal_check_data/.
"""
None

In [2]:
# IMPORT GENERAL PACKAGES
import pandas as pd
import numpy as np
# IMPORT PLOTLY
import plotly.express as px
# IMPORT THE SHARED CONFIGURATION, THE EXPERIMENT CONFIGURATION AND THE PATHS
from so import config, paths
from experiments.exp02_stop_reentry import config as exp_config
# IMPORT DATA LOADING, EXECUTION AND DAILY FEATURE FUNCTIONS
from so.core.raw_data import get_complete_ohlcv_pdf
from so.core.trade_execution import get_ohlcv_array_dict
from so.features.daily_features import get_daily_feature_pdf, get_daily_train_pdf
# IMPORT LOCAL FILE MANAGEMENT FUNCTIONS
from so.core.local_file_management import write_csv_file_to_path
# IMPORT THE EXPERIMENT FUNCTIONS
from experiments.exp02_stop_reentry.signal_check import get_daily_signal_check_pdf, get_daily_signal_check_verdict_dict
from experiments.exp02_stop_reentry.walk_forward import get_fold_pdf
# IMPORT TRIAL LOG FUNCTIONS
from so.core.trial_log import get_config_hash_str, log_trial_dict, read_trial_log_pdf, get_trial_count_pdf, print_previous_trial_warning

In [3]:
# DISPLAY THE EXPERIMENT, THE PROTOCOL VERSION AND THE CONFIGURATION HASH (SHARED + EXPERIMENT CONFIG)
print(f"Experiment:\t{exp_config.EXPERIMENT_NAME}\tProtocol:\t{exp_config.PROTOCOL_VERSION}\tConfig Hash:\t{get_config_hash_str(exp_config)}")
# DEFINE THE OUTPUT FOLDER OF THIS STEP
output_path_str = paths.get_experiment_data_path_str(exp_config.EXPERIMENT_NAME, "step02_signal_check_data")
print(f"Output folder:\t{output_path_str}")
# CALL FUNCTION TO WARN ABOUT EARLIER ENTRIES OF THIS STAGE (A RERUN ADDS TRIALS)
previous_trial_count = print_previous_trial_warning(exp_config, "signal_check")

Experiment:	exp02_stop_reentry	Protocol:	1.0	Config Hash:	53a86cd30f
Output folder:	C:/Users/nico/Desktop/stock_overflow_data/store04_experiments/exp02_stop_reentry/step02_signal_check_data/
ℹ️ No earlier 'signal_check' entry of exp02_stop_reentry in the trial log (config hash 53a86cd30f).


In [4]:
# CALL FUNCTION TO READ THE COMPLETE MINUTE BARS (RAW FOLDER OF so.paths)
complete_ohlcv_pdf = get_complete_ohlcv_pdf()
# CALL FUNCTION TO CONVERT THE COMPLETE DATA TO ARRAYS (USED BY THE SIMULATOR)
ohlcv_array_dict = get_ohlcv_array_dict(complete_ohlcv_pdf)
# CALL FUNCTION TO BUILD THE DAILY FEATURES AND TARGET (SECONDS; pipeline/step06 SAVES THE SAME TABLE AS A RECORD)
daily_pdf = get_daily_feature_pdf(ohlcv_array_dict)
# DISPLAY INFORMATION
print(f"min timestamp:\t{complete_ohlcv_pdf.timestamp.min()}")
print(f"max timestamp:\t{complete_ohlcv_pdf.timestamp.max()}")
print(f"Sessions:\t{len(daily_pdf):,}\tLabelled:\t{int(daily_pdf['y_fwd_positive'].notna().sum()):,}")

min timestamp:	2005-01-03 09:30:00-05:00
max timestamp:	2026-08-13 15:59:00-04:00
Sessions:	5,436	Labelled:	5,416


In [5]:
# CALL FUNCTION TO GET THE FOLDS
fold_pdf = get_fold_pdf(ohlcv_array_dict["session_date_list"])
# COLLECT THE POOLED FOLDS (THE MOST RECENT SELECTION_POOLED_QUARTER_COUNT FOLDS)
pooled_fold_pdf = fold_pdf.iloc[-exp_config.SELECTION_POOLED_QUARTER_COUNT:]
first_pooled_fold_row = pooled_fold_pdf.iloc[0]
# DEFINE THE TRAINING WINDOW (10 YEARS, ENDING BEFORE THE FIRST POOLED VALIDATION QUARTER)
train_start, train_end = first_pooled_fold_row["train_start_10y"], first_pooled_fold_row["train_end"]
# CALL FUNCTION TO READ THE TRAINING ROWS AND THE POOLED VALIDATION ROWS
signal_train_pdf = get_daily_train_pdf(daily_pdf, train_start, train_end)
signal_valid_pdf = pd.concat([get_daily_train_pdf(daily_pdf, row["valid_start"], row["valid_end"]) for _, row in pooled_fold_pdf.iterrows()], ignore_index=True)
# DISPLAY INFORMATION
print(f"Train:\t\t{train_start} -> {train_end}\t({len(signal_train_pdf):,} rows, base rate {signal_train_pdf['y_fwd_positive'].mean():.2%})")
for _, row in pooled_fold_pdf.iterrows():
    print(f"Validation:\t{row['valid_start']} -> {row['valid_end']}\t(fold {row['fold_id']})")
print(f"Pooled validation rows:\t{len(signal_valid_pdf):,} (base rate {signal_valid_pdf['y_fwd_positive'].mean():.2%})")
print(f"Latest test window (not read):\t{fold_pdf['test_start'].iloc[-1]} -> {fold_pdf['test_end'].iloc[-1]}")

Train:		2015-03-20 -> 2025-03-19	(2,515 rows, base rate 66.32%)
Validation:	2025-04-17 -> 2025-07-16	(fold 40)
Validation:	2025-07-17 -> 2025-10-16	(fold 41)
Validation:	2025-10-16 -> 2026-01-15	(fold 42)
Validation:	2026-01-16 -> 2026-04-15	(fold 43)
Pooled validation rows:	250 (base rate 77.60%)
Latest test window (not read):	2026-05-14 -> 2026-08-13


In [6]:
# CALL FUNCTION TO RUN THE SIGNAL CHECK (PROTOCOL MONTH MINIMUM FROM THE EXPERIMENT CONFIG)
signal_check_pdf = get_daily_signal_check_pdf(signal_train_pdf, signal_valid_pdf)
# CALL FUNCTION TO GET THE VERDICT
signal_verdict_dict = get_daily_signal_check_verdict_dict(signal_check_pdf)
# DISPLAY INFORMATION
for key_str, value in signal_verdict_dict.items():
    print(f"{key_str}:\t{value}")

Feature:	prev_day_return_pct	[1/16]
Feature:	prev_day_range_pct	[2/16]
Feature:	intraday_return_pct	[3/16]
Feature:	daily_volatility	[4/16]
Feature:	ath_drawdown_pct	[5/16]
Feature:	return_5d	[6/16]
Feature:	return_20d	[7/16]
Feature:	return_60d	[8/16]
Feature:	return_120d	[9/16]
Feature:	return_250d	[10/16]
Feature:	ma50_dist_pct	[11/16]
Feature:	ma200_dist_pct	[12/16]
Feature:	volatility_ratio_20_60	[13/16]
Feature:	high250_dist_pct	[14/16]
Feature:	high60_dist_pct	[15/16]
Feature:	sessions_since_high60	[16/16]
test_count:	80
train_pass_count:	0
expected_chance_pass_count:	4.0
signal_count:	0
signal_feature_list:	[]
verdict:	STOP


In [7]:
"""
Lift = bin rate - base rate of the window. Training lift on the x axis, validation lift on the y axis: real information
sits in the bottom-left or top-right quadrant on BOTH axes (same sign), far from 0.
"""
# CALCULATE THE LIFTS
signal_check_pdf["train_lift"] = signal_check_pdf["train_rate"] - signal_check_pdf["train_base_rate"]
signal_check_pdf["valid_lift"] = signal_check_pdf["valid_rate"] - signal_check_pdf["valid_base_rate"]
# CREATE A FIGURE OF TRAINING LIFT vs VALIDATION LIFT
fig = px.scatter(signal_check_pdf[signal_check_pdf["tested"]], x="train_lift", y="valid_lift", color="feature", symbol="signal",
                 hover_data=["bin", "train_row_count", "valid_row_count", "train_month_count", "valid_month_count"], template="plotly_dark", width=1100, height=650,
                 title="Training Lift vs Pooled Validation Lift (rate of positive 20-session returns - base rate)")
fig.add_hline(y=0, line_dash="dash", line_color="white")
fig.add_vline(x=0, line_dash="dash", line_color="white")
fig.update_layout(xaxis_tickformat=".0%", yaxis_tickformat=".0%")
fig.show()

In [8]:
# PREVIEW THE SIGNAL BINS
display(signal_check_pdf[signal_check_pdf["signal"]])
# PREVIEW THE BINS WITH THE LARGEST TRAINING LIFT (BOTH SIDES)
signal_check_pdf[signal_check_pdf["tested"]].reindex(signal_check_pdf[signal_check_pdf["tested"]]["train_lift"].abs().sort_values(ascending=False).index) \
    [["feature", "bin", "train_row_count", "train_month_count", "train_rate", "train_rate_ci_low", "train_rate_ci_high", "train_base_rate",
      "valid_row_count", "valid_month_count", "valid_rate", "valid_rate_ci_low", "valid_rate_ci_high", "valid_base_rate", "signal"]].head(20).round(4)

,feature,bin,train_row_count,train_month_count,train_rate,train_rate_ci_low,train_rate_ci_high,train_base_rate,train_mean_fwd_return,valid_row_count,...,valid_base_rate,valid_mean_fwd_return,tested,train_above,train_below,valid_above,valid_below,signal,train_lift,valid_lift


,feature,bin,train_row_count,train_month_count,train_rate,train_rate_ci_low,train_rate_ci_high,train_base_rate,valid_row_count,valid_month_count,valid_rate,valid_rate_ci_low,valid_rate_ci_high,valid_base_rate,signal
40,return_120d,"(0.0424, 0.0811]",503,68,0.7435,0.6371,0.8268,0.6632,49.0,7.0,0.5714,0.1041,0.9643,0.776,False
16,daily_volatility,"(0.00891, 0.0126]",503,60,0.5885,0.4458,0.7282,0.6632,45.0,8.0,0.8667,0.6176,0.9767,0.776,False
41,return_120d,"(0.0811, 0.128]",503,65,0.6024,0.4863,0.7130,0.6632,48.0,5.0,0.6458,0.2222,1.0000,0.776,False
77,sessions_since_high60,"(39.0, inf]",496,41,0.7137,0.6032,0.8189,0.6632,40.0,4.0,1.0000,1.0000,1.0000,0.776,False
49,return_250d,"(0.226, inf]",503,35,0.7117,0.5982,0.8085,0.6632,8.0,1.0,1.0000,1.0000,1.0000,0.776,False
57,ma200_dist_pct,"(-inf, 0.000113]",503,42,0.6163,0.4820,0.7432,0.6632,28.0,4.0,1.0000,1.0000,1.0000,0.776,False
42,return_120d,"(-0.012, 0.0424]",503,59,0.6203,0.5070,0.7263,0.6632,47.0,6.0,0.9149,0.7647,1.0000,0.776,False
10,intraday_return_pct,"(0.00175, 0.0052]",503,115,0.7058,0.6281,0.7656,0.6632,51.0,12.0,0.8039,0.5681,0.9608,0.776,False
21,ath_drawdown_pct,"(-0.0419, -0.016]",503,72,0.7058,0.6092,0.8009,0.6632,65.0,10.0,0.7692,0.4473,1.0000,0.776,False
55,ma200_dist_pct,"(0.0398, 0.0638]",503,65,0.7058,0.5874,0.8042,0.6632,25.0,7.0,0.4400,0.0769,1.0000,0.776,False


In [9]:
# SAVE THE SIGNAL CHECK RESULTS
write_csv_file_to_path(signal_check_pdf, f"{output_path_str}daily_signal_check_data.csv", "W")
# LOG THE TRIAL (THE LATEST TEST WINDOW WAS NOT READ)
signal_trial_dict = log_trial_dict(exp_config, "signal_check", {"train_window": "10y", "train_start": train_start, "train_end": train_end,
                                            "valid_fold_id_list": pooled_fold_pdf["fold_id"].tolist(), "feature_count": len(config.DAILY_FEATURE_COL_STR_LIST),
                                            "min_month_count": exp_config.DAILY_SIGNAL_CHECK_MIN_MONTH_COUNT},
               signal_verdict_dict, False, pooled_fold_pdf.iloc[-1], note_str_in="exp02 step02 daily signal check (stopping rule, two-sided, base rate, month minimum)")
print(f"Logged 1 signal-check entry (config hash {get_config_hash_str(exp_config)})")

ℹ️ Saving In Overwrite Mode.	(💾 In Local File Storage)
⚠️ File:	'C:/Users/nico/Desktop/stock_overflow_data/store04_experiments/exp02_stop_reentry/step02_signal_check_data/daily_signal_check_data.csv' Not Found!	(💾 In Local File Storage)
✅ File:	'C:/Users/nico/Desktop/stock_overflow_data/store04_experiments/exp02_stop_reentry/step02_signal_check_data/daily_signal_check_data.csv' Has Been Saved!
Logged 1 signal-check entry (config hash 53a86cd30f)


In [10]:
# CALL FUNCTION TO READ THE TRIAL LOG
trial_log_pdf = read_trial_log_pdf()
# DISPLAY THE NUMBER OF TRIALS PER EXPERIMENT AND STAGE (MULTIPLE-TESTING RECORD)
get_trial_count_pdf(trial_log_pdf)

,experiment,stage,trial_count,config_count,test_touched_count,first_logged_ts,last_logged_ts
0,exp02_stop_reentry,exploration,14,1,0,2026-10-05T14:47:50,2026-10-05T14:47:50
1,exp02_stop_reentry,signal_check,1,1,0,2026-10-05T14:48:41,2026-10-05T14:48:41
